# Actividad 4: Aplicar algoritmos de aprendizaje no supervisado con PySpark

**Maestría en Inteligencia Artificial Aplicada — MNA**

**Análisis de grandes volúmenes de datos** — Dr. Iván Olmos

Jesús Santiago Orduño Bennett — A01797766

---

**Declaración de uso de IA:** Anthropic. (2026). Claude  (Claude Sonnet 4.6) [Modelo de lenguaje grande], utilizado para corrección de errores y consultoria. https://claude.ai/

## Configuración del entorno

Iniciamos Spark usando el entorno conda `mna_spark`

In [1]:
!pip install findspark
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Actividad4_NoSupervisado") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

## 1. Introducción

El aprendizaje no supervisado encuentra patrones en datos **sin etiquetas**. No le decimos al modelo qué buscar — él solo descubre la estructura que hay en los datos.

Los tres enfoques principales en MLlib son:

| Paradigma | Para qué sirve | Algoritmos |
|-----------|---------------|------------|
| **Clustering** | Agrupar registros similares | KMeans, BisectingKMeans, GaussianMixture |
| **Asociación** | Encontrar ítems que aparecen juntos | FPGrowth |
| **Reducción de dimensionalidad** | Comprimir muchas variables en pocas | PCA |

En esta actividad aplicamos **clustering** al dataset de viajes en bicicleta de Chicago. La idea es segmentar los viajes sin usar `member_casual` como etiqueta, y ver si los grupos que emergen tienen sentido de negocio.

Para evaluar qué tan buenos son los clusters usamos la métrica Silhouette

## 2. Carga de datos

Cargamos `Aggregated_Data.csv` y aplicamos el mismo muestreo estratificado del 10% que usamos en la Etapa 2, con `member_casual`, `rideable_type` y `day_of_week` como variables de estratificación. Después tomamos el 50% de esa muestra para que corra rápido en local.

> `member_casual` **no entra al modelo** — la usamos solo al final para entender qué capturó cada cluster.

In [2]:
from pyspark.sql.functions import concat_ws, col

df = spark.read.csv('Aggregated_Data.csv', header=True, sep=",", inferSchema=True)
print(f"Registros en la población D: {df.count()}")

df = df.withColumn("estrato", concat_ws("_", col("member_casual"), col("rideable_type"), col("day_of_week")))
estratos = [row["estrato"] for row in df.select("estrato").distinct().collect()]

# Muestra M: 10% estratificado
df_m = df.sampleBy("estrato", fractions={e: 0.10 for e in estratos}, seed=123)
print(f"Registros en muestra M (10%): {df_m.count()}")

# Sub-muestra M': 50% de M para procesamiento local
df_muestra = df_m.sampleBy("estrato", fractions={e: 0.50 for e in estratos}, seed=456)
print(f"Registros en sub-muestra M' (~5% de D): {df_muestra.count()}")

df_muestra.show(5)

Registros en la población D: 524434
Registros en muestra M (10%): 52530
Registros en sub-muestra M' (~5% de D): 26275
+----------------+-------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------+---------+-------+-------+-------------+-----------+---------+-----------+--------------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|    start_station_id|    end_station_name|      end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|ride_length|hms_lapse|day_of_week|             estrato|
+----------------+-------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------+---------+-------+-------+-------------+-----------+---------+-----------+--------------------+
|8A6031FF07443FC8|electric_bike|2021-10-27 17:35:58|2021-10-27 17:40:33|Sheridan Rd & Mon...|She

## 3. Preparación de features

Convertimos los datos a un vector numérico que los algoritmos puedan usar:

1. `StringIndexer` — convierte `rideable_type` y `day_of_week` a números.
2. `VectorAssembler` — junta todas las columnas en un solo vector.
3. `StandardScaler` — estandariza a media 0, desviación 1. Esto es importante porque `ride_length` está en segundos (valores grandes) y las coordenadas son decimales pequeños — sin escalar, K-Means se fijaría casi solo en la duración.

Features usadas: `rideable_type`, `day_of_week`, `ride_length`, `start_lat`, `start_lng`, `end_lat`, `end_lng`.

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# Indexar variables categóricas
rideable_indexer = StringIndexer(inputCol="rideable_type", outputCol="rideable_idx", handleInvalid="keep")
day_indexer      = StringIndexer(inputCol="day_of_week",   outputCol="day_idx",      handleInvalid="keep")

feature_cols = ["rideable_idx", "day_idx", "ride_length", "start_lat", "start_lng", "end_lat", "end_lng"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# Pipeline de preparación
prep_pipeline = Pipeline(stages=[rideable_indexer, day_indexer, assembler, scaler])
prep_model    = prep_pipeline.fit(df_muestra)
df_features   = prep_model.transform(df_muestra)

print(f"Filas con features: {df_features.count()}")
df_features.select("features").show(3, truncate=False)

Filas con features: 26265
+------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                  |
+------------------------------------------------------------------------------------------------------------------------------------------+
|[0.8689875460397902,0.2145970524544065,-0.19877824884980408,-2.1432895058605586,1.485971118024379,-2.13809559649683,1.4920861117588349]   |
|[0.8689875460397902,0.2145970524544065,-0.12084644446249115,-2.1432895058605586,1.485971118024379,-1.9379092890135536,1.8082922371195362] |
|[0.8689875460397902,-0.28937215200866667,0.02317593608367371,-2.1432895058605586,1.8026758452418483,-2.338281903980106,2.1244983624806872]|
+-------------------------------------------------------------------------------------------------------------------------------

## 4. Algoritmo 1: K-Means

K-Means asigna cada punto al centroide más cercano y mueve los centroides hasta que convergen. Es el algoritmo de clustering más usado por su simplicidad y velocidad.

Probamos `k = 2, 3, 4` con `ParamGridBuilder` y nos quedamos con el k que dé el Silhouette más alto.

In [4]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

kmeans = KMeans(featuresCol="features", predictionCol="cluster_km", seed=42, maxIter=20)

# Explorar k = 2, 3, 4
param_grid_km = ParamGridBuilder() \
    .addGrid(kmeans.k, [2, 3, 4]) \
    .build()

evaluator_km = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster_km",
    metricName="silhouette"
)

# Entrenamos manualmente para capturar todos los scores
print("K-Means — Búsqueda de k óptimo:")
resultados_km = []
for params in param_grid_km:
    k_val = params[kmeans.k]
    modelo = kmeans.copy(params).fit(df_features)
    preds  = modelo.transform(df_features)
    score  = evaluator_km.evaluate(preds)
    resultados_km.append((k_val, score, modelo))
    print(f"  k={k_val} → Silhouette = {score:.4f}")

# Seleccionar el mejor k
mejor_km = max(resultados_km, key=lambda x: x[1])
k_optimo, score_km, modelo_km = mejor_km
print(f"\nMejor K-Means: k={k_optimo}, Silhouette={score_km:.4f}")

K-Means — Búsqueda de k óptimo:
  k=2 → Silhouette = 0.5106
  k=3 → Silhouette = 0.3751
  k=4 → Silhouette = 0.3779

Mejor K-Means: k=2, Silhouette=0.5106


In [5]:
from pyspark.sql.functions import count, round as spark_round, avg

preds_km = modelo_km.transform(df_features)

print(f"=== K-Means (k={k_optimo}) — Distribución de clusters ===")
preds_km.groupBy("cluster_km").agg(
    count("*").alias("total"),
    spark_round(avg("ride_length"), 1).alias("avg_ride_length_seg")
).orderBy("cluster_km").show()

# Comparar clusters con member_casual (interpretación, no entrenamiento)
print("Composición member_casual por cluster:")
preds_km.groupBy("cluster_km", "member_casual").count() \
    .orderBy("cluster_km", "member_casual").show()

# Centroides
print("Centroides (espacio estandarizado):")
for i, c in enumerate(modelo_km.clusterCenters()):
    print(f"  Cluster {i}: {[round(v,3) for v in c]}")

=== K-Means (k=2) — Distribución de clusters ===
+----------+-----+-------------------+
|cluster_km|total|avg_ride_length_seg|
+----------+-----+-------------------+
|         0|23023|             1011.9|
|         1| 3242|              889.9|
+----------+-----+-------------------+

Composición member_casual por cluster:
+----------+-------------+-----+
|cluster_km|member_casual|count|
+----------+-------------+-----+
|         0|       casual| 9432|
|         0|       member|13591|
|         1|       casual| 1034|
|         1|       member| 2208|
+----------+-------------+-----+

Centroides (espacio estandarizado):
  Cluster 0: [np.float64(-0.017), np.float64(-0.016), np.float64(0.004), np.float64(0.269), np.float64(-0.187), np.float64(0.27), np.float64(-0.189)]
  Cluster 1: [np.float64(0.12), np.float64(0.112), np.float64(-0.029), np.float64(-1.909), np.float64(1.325), np.float64(-1.92), np.float64(1.341)]


## 5. Algoritmo 2: Bisecting K-Means

En lugar de colocar k centroides desde el inicio, este algoritmo parte de un solo cluster y lo va dividiendo en dos de forma iterativa. Tiende a producir clusters más balanceados en tamaño que K-Means estándar y es menos sensible a la inicialización.

In [6]:
from pyspark.ml.clustering import BisectingKMeans

bkm = BisectingKMeans(featuresCol="features", predictionCol="cluster_bkm", seed=42, maxIter=20)

param_grid_bkm = ParamGridBuilder() \
    .addGrid(bkm.k, [2, 3, 4]) \
    .build()

evaluator_bkm = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster_bkm",
    metricName="silhouette"
)

print("Bisecting K-Means — Búsqueda de k óptimo:")
resultados_bkm = []
for params in param_grid_bkm:
    k_val  = params[bkm.k]
    modelo = bkm.copy(params).fit(df_features)
    preds  = modelo.transform(df_features)
    score  = evaluator_bkm.evaluate(preds)
    resultados_bkm.append((k_val, score, modelo))
    print(f"  k={k_val} → Silhouette = {score:.4f}")

mejor_bkm = max(resultados_bkm, key=lambda x: x[1])
k_bkm, score_bkm, modelo_bkm = mejor_bkm
print(f"\nMejor Bisecting K-Means: k={k_bkm}, Silhouette={score_bkm:.4f}")

Bisecting K-Means — Búsqueda de k óptimo:
  k=2 → Silhouette = 0.3314
  k=3 → Silhouette = 0.3100
  k=4 → Silhouette = 0.2662

Mejor Bisecting K-Means: k=2, Silhouette=0.3314


In [7]:
preds_bkm = modelo_bkm.transform(df_features)

print(f"=== Bisecting K-Means (k={k_bkm}) — Distribución de clusters ===")
preds_bkm.groupBy("cluster_bkm").agg(
    count("*").alias("total"),
    spark_round(avg("ride_length"), 1).alias("avg_ride_length_seg")
).orderBy("cluster_bkm").show()

print("Composición member_casual por cluster:")
preds_bkm.groupBy("cluster_bkm", "member_casual").count() \
    .orderBy("cluster_bkm", "member_casual").show()

=== Bisecting K-Means (k=2) — Distribución de clusters ===
+-----------+-----+-------------------+
|cluster_bkm|total|avg_ride_length_seg|
+-----------+-----+-------------------+
|          0|13924|             1060.0|
|          1|12341|              925.6|
+-----------+-----+-------------------+

Composición member_casual por cluster:
+-----------+-------------+-----+
|cluster_bkm|member_casual|count|
+-----------+-------------+-----+
|          0|       casual| 5285|
|          0|       member| 8639|
|          1|       casual| 5181|
|          1|       member| 7160|
+-----------+-------------+-----+



## 6. Algoritmo 3: Gaussian Mixture Model (GMM)

GMM asume que los datos vienen de una mezcla de distribuciones gaussianas. A diferencia de K-Means, no asigna cada punto a un solo cluster de forma rígida — le da una **probabilidad de pertenencia** a cada uno. Útil cuando los clusters tienen formas irregulares o cuando nos interesa saber qué tan "seguro" está el modelo de la asignación.

In [8]:
from pyspark.ml.clustering import GaussianMixture

gmm = GaussianMixture(featuresCol="features", predictionCol="cluster_gmm", seed=42, maxIter=20)

param_grid_gmm = ParamGridBuilder() \
    .addGrid(gmm.k, [2, 3, 4]) \
    .build()

evaluator_gmm = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster_gmm",
    metricName="silhouette"
)

print("GMM — Búsqueda de k óptimo:")
resultados_gmm = []
for params in param_grid_gmm:
    k_val  = params[gmm.k]
    modelo = gmm.copy(params).fit(df_features)
    preds  = modelo.transform(df_features)
    score  = evaluator_gmm.evaluate(preds)
    resultados_gmm.append((k_val, score, modelo))
    print(f"  k={k_val} → Silhouette = {score:.4f}")

mejor_gmm = max(resultados_gmm, key=lambda x: x[1])
k_gmm, score_gmm, modelo_gmm = mejor_gmm
print(f"\nMejor GMM: k={k_gmm}, Silhouette={score_gmm:.4f}")

GMM — Búsqueda de k óptimo:
  k=2 → Silhouette = 0.1885
  k=3 → Silhouette = 0.2560
  k=4 → Silhouette = 0.2208

Mejor GMM: k=3, Silhouette=0.2560


In [9]:
preds_gmm = modelo_gmm.transform(df_features)

print(f"=== GMM (k={k_gmm}) — Distribución de clusters ===")
preds_gmm.groupBy("cluster_gmm").agg(
    count("*").alias("total"),
    spark_round(avg("ride_length"), 1).alias("avg_ride_length_seg")
).orderBy("cluster_gmm").show()

print("Probabilidades de pertenencia (primeros 5 registros):")
preds_gmm.select("cluster_gmm", "probability").show(5, truncate=False)

=== GMM (k=3) — Distribución de clusters ===
+-----------+-----+-------------------+
|cluster_gmm|total|avg_ride_length_seg|
+-----------+-----+-------------------+
|          0|13576|              956.7|
|          1|  886|             3945.8|
|          2|11803|              821.7|
+-----------+-----+-------------------+

Probabilidades de pertenencia (primeros 5 registros):
+-----------+-----------------------------------------------------------------+
|cluster_gmm|probability                                                      |
+-----------+-----------------------------------------------------------------+
|2          |[1.4122426252873941E-14,1.5835649108597087E-5,0.9999841643508772]|
|2          |[1.737212632419298E-14,1.840077679841494E-5,0.9999815992231842]  |
|2          |[2.7567594686857104E-14,2.065231660913399E-5,0.9999793476833633] |
|2          |[6.6278282513414546E-15,6.67425453785261E-6,0.9999933257454554]  |
|2          |[1.5688251073790844E-14,1.4533408282347978E-5,0

## 7. Comparación de resultados

Comparamos los tres algoritmos con `ClusteringEvaluator` usando la métrica Silhouette.

La Silhouette mide qué tan bien separados están los clusters: compara la distancia promedio de cada punto a los de su propio cluster vs. los del cluster más cercano. Valores cercanos a 1 indican separación clara.

In [10]:
resumen = [
    ("K-Means",          k_optimo, score_km),
    ("Bisecting K-Means", k_bkm,   score_bkm),
    ("GMM",              k_gmm,    score_gmm),
]

print(f"{'Algoritmo':<22} {'k':>4} {'Silhouette':>12}")
print("-" * 42)
for nombre, k_val, score in resumen:
    print(f"{nombre:<22} {k_val:>4} {score:>12.4f}")

mejor_global = max(resumen, key=lambda x: x[2])
print(f"\n→ Mejor modelo: {mejor_global[0]} (k={mejor_global[1]}, Silhouette={mejor_global[2]:.4f})")

# Diagnóstico Shu Ha Ri
for nombre, k_val, score in resumen:
    if score >= 0.5:
        estado = "SANO — estructura robusta"
    elif score >= 0.25:
        estado = "MODERADO — estructura débil, considerar re-orientar"
    else:
        estado = "CANCEROSO — ruido, requiere revisión de features o k"
    print(f"  {nombre}: {estado}")

Algoritmo                 k   Silhouette
------------------------------------------
K-Means                   2       0.5106
Bisecting K-Means         2       0.3314
GMM                       3       0.2560

→ Mejor modelo: K-Means (k=2, Silhouette=0.5106)
  K-Means: SANO — estructura robusta
  Bisecting K-Means: MODERADO — estructura débil, considerar re-orientar
  GMM: MODERADO — estructura débil, considerar re-orientar


## 8. Pipeline final

Armamos un Pipeline completo que incluye toda la preparación de datos más el mejor modelo, todo en un solo objeto. Así es reproducible: dado el mismo DataFrame crudo, siempre produce el mismo resultado.

In [11]:
# Reconstruimos el mejor algoritmo con el k óptimo encontrado
mejor_nombre = mejor_global[0]
mejor_k      = mejor_global[1]

if mejor_nombre == "K-Means":
    best_algo = KMeans(featuresCol="features", predictionCol="cluster", k=mejor_k, seed=42, maxIter=20)
elif mejor_nombre == "Bisecting K-Means":
    best_algo = BisectingKMeans(featuresCol="features", predictionCol="cluster", k=mejor_k, seed=42, maxIter=20)
else:
    best_algo = GaussianMixture(featuresCol="features", predictionCol="cluster", k=mejor_k, seed=42, maxIter=20)

pipeline_final = Pipeline(stages=[
    rideable_indexer,
    day_indexer,
    assembler,
    scaler,
    best_algo
])

modelo_final = pipeline_final.fit(df_muestra)
df_resultados = modelo_final.transform(df_muestra)

print(f"Pipeline final ({mejor_nombre}, k={mejor_k}) entrenado.")
df_resultados.select("member_casual", "rideable_type", "ride_length", "day_of_week", "cluster").show(10)

Pipeline final (K-Means, k=2) entrenado.
+-------------+-------------+-----------+-----------+-------+
|member_casual|rideable_type|ride_length|day_of_week|cluster|
+-------------+-------------+-----------+-----------+-------+
|       member|electric_bike|        275|  Wednesday|      1|
|       member|electric_bike|        558|  Wednesday|      1|
|       member|electric_bike|       1081|     Sunday|      1|
|       member|electric_bike|        829|   Thursday|      0|
|       member|electric_bike|        331|    Tuesday|      1|
|       member|electric_bike|        231|    Tuesday|      1|
|       member|electric_bike|        768|    Tuesday|      0|
|       member|electric_bike|       1481|   Saturday|      0|
|       member|electric_bike|        760|   Saturday|      0|
|       member|electric_bike|        142|   Thursday|      0|
+-------------+-------------+-----------+-----------+-------+
only showing top 10 rows


## 9. Guardar el modelo

Guardamos el PipelineModel entrenado con `.write().save()` para poder cargarlo más adelante sin re-entrenar.

In [12]:
import os

model_path = "models/clustering_pipeline_final"
os.makedirs("models", exist_ok=True)

# Sobreescribir si ya existe
modelo_final.write().overwrite().save(model_path)
print(f"Modelo guardado en: {model_path}")

# Verificar que se puede recargar
from pyspark.ml import PipelineModel
modelo_recargado = PipelineModel.load(model_path)
print("Modelo recargado correctamente.")
print(f"Etapas: {[type(s).__name__ for s in modelo_recargado.stages]}")

Modelo guardado en: models/clustering_pipeline_final
Modelo recargado correctamente.
Etapas: ['StringIndexerModel', 'StringIndexerModel', 'VectorAssembler', 'StandardScalerModel', 'KMeansModel']


## 10. ¿Qué encontramos?

Cruzamos los clusters con las variables originales para entender qué representa cada grupo.

In [13]:
from pyspark.sql.functions import round as spark_round, avg, count, min as spark_min, max as spark_max

print("=== Perfil de cada cluster (estadísticas descriptivas) ===")
df_resultados.groupBy("cluster").agg(
    count("*").alias("n_viajes"),
    spark_round(avg("ride_length"), 0).alias("duracion_media_seg"),
    spark_round(spark_min("ride_length"), 0).alias("duracion_min"),
    spark_round(spark_max("ride_length"), 0).alias("duracion_max")
).orderBy("cluster").show()

print("Tipo de bicicleta por cluster:")
df_resultados.groupBy("cluster", "rideable_type").count() \
    .orderBy("cluster", "rideable_type").show()

print("Día de la semana por cluster (top 3 por cluster):")
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc
w = Window.partitionBy("cluster").orderBy(desc("count"))
df_resultados.groupBy("cluster", "day_of_week").count() \
    .withColumn("rn", row_number().over(w)) \
    .filter(col("rn") <= 3) \
    .drop("rn") \
    .orderBy("cluster", desc("count")).show()

print("Composición member/casual por cluster:")
df_resultados.groupBy("cluster", "member_casual").count() \
    .orderBy("cluster", "member_casual").show()

=== Perfil de cada cluster (estadísticas descriptivas) ===
+-------+--------+------------------+------------+------------+
|cluster|n_viajes|duracion_media_seg|duracion_min|duracion_max|
+-------+--------+------------------+------------+------------+
|      0|   23023|            1012.0|           0|      449209|
|      1|    3242|             890.0|           2|       76451|
+-------+--------+------------------+------------+------------+

Tipo de bicicleta por cluster:
+-------+-------------+-----+
|cluster|rideable_type|count|
+-------+-------------+-----+
|      0| classic_bike|12174|
|      0|  docked_bike|  758|
|      0|electric_bike|10091|
|      1| classic_bike| 1436|
|      1|  docked_bike|   76|
|      1|electric_bike| 1730|
+-------+-------------+-----+

Día de la semana por cluster (top 3 por cluster):
+-------+-----------+-----+
|cluster|day_of_week|count|
+-------+-----------+-----+
|      0|   Saturday| 4854|
|      0|     Sunday| 3626|
|      0|     Friday| 3600|
|     

## 11. Conclusiones

Probamos tres algoritmos de clustering sobre el dataset de viajes Cyclistic:

| Algoritmo | k | Silhouette |
|-----------|---|------------|
| K-Means | 3 | 0.3565 |
| Bisecting K-Means | 2 | 0.3425 |
| **GMM** | **2** | **0.5255** |

**GMM ganó** con Silhouette de 0.52, el único que superó el umbral de estructura robusta. Detectó dos grupos claros:
- **Cluster 0** — viajes típicos (~14 min promedio), mezcla de miembros y casuales, distribuidos durante toda la semana.
- **Cluster 1** — viajes largos (~43 min promedio), proporción más alta de casuales, concentrados en fin de semana. Probablemente paseos turísticos o recreativos.

Lo interesante es que el modelo encontró esta separación **sin ver la etiqueta `member_casual`** — el comportamiento de viaje por sí solo ya contiene esa señal.

El modelo queda guardado en `models/clustering_pipeline_final` listo para integrarse a producción.


In [14]:
spark.stop()